# Chapter 12 - Neural Networks that Write Like Shakespeare: Recurrent Layers for Variable-Length Data

## Summary
Sentences have *order* and *variable length*; a bag-of-words sum ignores both.
- **Problem with summing embeddings:** `"red cat chased dog"` and `"dog chased red cat"` produce the same vector - order is lost.
- **Identity-matrix trick:** summing embeddings in order, but passing the running state through a (here identity) matrix at each step, gives an order-aware "sentence embedding".
- **Recurrent Neural Network (RNN):** at each time step the network combines the *current input* with the *previous hidden state*: `h_t = f(W_xh x_t + W_hh h_{t-1})`. The hidden state is a *memory* of everything seen so far.
- **Backpropagation Through Time (BPTT):** unroll the network across time steps and backpropagate through all of them; weights are *shared* across steps so their gradients are summed.
- **Truncated BPTT:** only backpropagate over a fixed window (e.g. 25 steps) to keep it cheap and stable.
- **Vanishing / exploding gradients:** repeated multiplication by `W_hh` shrinks or blows up the gradient; clipping and (next) LSTMs help.
- **Character-level language model:** predict the next character -> sampling from it generates text 'like Shakespeare'.

## Theory
| Concept | Meaning |
|---|---|
| Hidden state `h_t` | Fixed-size vector summarising the sequence so far |
| Shared weights | Same `W_xh, W_hh, W_hy` at each step => handles any length |
| `tanh` | Keeps the hidden state in (-1, 1) |
| Softmax + cross-entropy | Turns outputs into next-character probabilities |
| BPTT | Chain rule through time: `dh_t = W_hy^T dy_t + W_hh^T dh_{t+1} * tanh'` |
| Gradient clipping | `np.clip(grad, -5, 5)` prevents explosion |
| Sampling with temperature | Draw next char from softmax(logits/T); low T = safe, high T = creative |

Vanishing gradients: if the largest eigenvalue of `W_hh` < 1 the gradient shrinks exponentially with distance, so plain RNNs forget long-range information.

In [1]:
# If packages are missing in this kernel, uncomment:
# %pip install numpy scikit-learn matplotlib
import numpy as np

### 1) Why order matters: identity-matrix sentence embedding

In [2]:
np.random.seed(1)
emb = {w: np.random.randn(4) for w in ["red", "cat", "chased", "dog"]}
def bag(sentence): return sum(emb[w] for w in sentence)
def ordered(sentence, W):
    h = np.zeros(4)
    for w in sentence: h = h.dot(W) + emb[w]     # state is transformed at each step
    return h
s1, s2 = ["red", "cat", "chased", "dog"], ["dog", "chased", "red", "cat"]
print("bag-of-words equal?     ", np.allclose(bag(s1), bag(s2)))
print("identity transition eq? ", np.allclose(ordered(s1, np.eye(4)), ordered(s2, np.eye(4))))
W = np.random.randn(4, 4) * 0.5
print("learned-style W equal?  ", np.allclose(ordered(s1, W), ordered(s2, W)))

bag-of-words equal?      True
identity transition eq?  True
learned-style W equal?   False


With an *identity* matrix order is still ignored (it's just summing), but with a **non-identity transition matrix** (which an RNN learns) different orders give different states.

### 2) A character-level RNN trained with truncated BPTT (pure NumPy)

In [3]:
text = ("to be, or not to be, that is the question: whether 'tis nobler in the mind to suffer "
        "the slings and arrows of outrageous fortune, or to take arms against a sea of troubles "
        "and by opposing end them. ") * 4
chars = sorted(set(text)); V = len(chars)
c2i = {c: i for i, c in enumerate(chars)}; i2c = {i: c for c, i in c2i.items()}
data = [c2i[c] for c in text]
print("text length", len(data), "| vocab", V)

text length 792 | vocab 26


In [4]:
np.random.seed(1)
H, seq_len, lr = 64, 25, 0.1
Wxh = np.random.randn(H, V) * 0.01
Whh = np.random.randn(H, H) * 0.01
Why = np.random.randn(V, H) * 0.01
bh, by = np.zeros((H, 1)), np.zeros((V, 1))

def loss_fn(inputs, targets, hprev):
    xs, hs, ps = {}, {-1: hprev.copy()}, {}
    loss = 0
    for t, ix in enumerate(inputs):                       # ---- forward through time
        xs[t] = np.zeros((V, 1)); xs[t][ix] = 1
        hs[t] = np.tanh(Wxh.dot(xs[t]) + Whh.dot(hs[t-1]) + bh)
        y = Why.dot(hs[t]) + by
        ps[t] = np.exp(y - y.max()); ps[t] /= ps[t].sum()
        loss += -np.log(ps[t][targets[t], 0])
    grads = [np.zeros_like(m) for m in (Wxh, Whh, Why, bh, by)]
    dWxh, dWhh, dWhy, dbh, dby = grads
    dh_next = np.zeros_like(hs[0])
    for t in reversed(range(len(inputs))):                # ---- backward through time (BPTT)
        dy = ps[t].copy(); dy[targets[t]] -= 1
        dWhy += dy.dot(hs[t].T); dby += dy
        dh = Why.T.dot(dy) + dh_next
        draw = (1 - hs[t] ** 2) * dh
        dbh += draw; dWxh += draw.dot(xs[t].T); dWhh += draw.dot(hs[t-1].T)
        dh_next = Whh.T.dot(draw)
    for g in grads: np.clip(g, -5, 5, out=g)              # gradient clipping
    return loss, grads, hs[len(inputs) - 1]

def sample(h, seed_ix, n, temperature=1.0):
    x = np.zeros((V, 1)); x[seed_ix] = 1; out = []
    for _ in range(n):
        h = np.tanh(Wxh.dot(x) + Whh.dot(h) + bh)
        y = (Why.dot(h) + by) / temperature
        p = np.exp(y - y.max()); p /= p.sum()
        ix = np.random.choice(V, p=p.ravel())
        x = np.zeros((V, 1)); x[ix] = 1; out.append(i2c[ix])
    return "".join(out)

mem = [np.zeros_like(m) for m in (Wxh, Whh, Why, bh, by)]   # Adagrad memory
params = [Wxh, Whh, Why, bh, by]
p, hprev, smooth = 0, np.zeros((H, 1)), -np.log(1.0 / V) * seq_len
for n in range(2501):
    if p + seq_len + 1 >= len(data): p, hprev = 0, np.zeros((H, 1))
    inputs, targets = data[p:p+seq_len], data[p+1:p+seq_len+1]
    loss, grads, hprev = loss_fn(inputs, targets, hprev)
    smooth = 0.999 * smooth + 0.001 * loss
    for prm, g, m in zip(params, grads, mem):
        m += g * g; prm -= lr * g / np.sqrt(m + 1e-8)       # Adagrad update
    p += seq_len
    if n % 500 == 0:
        print(f"iter {n:4d} loss {smooth:.3f}")
        print("   >", sample(hprev, inputs[0], 70).replace("\n", " "))

iter    0 loss 81.452
   > b'tr pberybiglkoiodaul .orshmrwtfw.ukpretb,qtufebiykbauopmn'e,ep'ue,,.
iter  500 loss 64.099
   > ptonut'tins to suag inst a sea of trogbe, 'tind to suag insteo th oe: 
iter 1000 loss 39.958
   > gs the slings and arrows of outrageous fortune, or to take arms agains
iter 1500 loss 24.531
   > , or to take arms against a sea of trous tune, or not to be, thathi n:
iter 2000 loss 15.046
   > powstamg, or to take arms against a sea of troubles and by opposing en
iter 2500 loss 9.242
   > or no ingtisunoflto the mht er sndthe question: whetherastat is the qu


In [5]:
print("temperature 0.5:", sample(np.zeros((H, 1)), c2i["t"], 100, 0.5))
print("temperature 1.2:", sample(np.zeros((H, 1)), c2i["t"], 100, 1.2))

temperature 0.5: o be, or not to be, that is the question: whether 'tis nobler in the mind to suffer the slings and a
temperature 1.2: o be, or not to be, that is the question: whether 'tis ny by hetosthebst the question: whether 'tis 


## Takeaways
- An RNN keeps a hidden state that is updated at every step, so it handles sequences of any length and respects order.
- Training = unroll through time and apply backprop (BPTT); gradient clipping tames exploding gradients.
- Plain RNNs struggle with long-term dependencies (vanishing gradients) - the motivation for the LSTM (Chapter 14).